# vForge — 04. Benchmark with vLLM

Run inference benchmarks on TPU and GPU. Captures throughput, latency, memory, and cost.
**Run twice** — once with TPU runtime, once with GPU — and aggregate the two JSON files.

## 1. Install

In [ ]:
!pip -q install vllm

## 2. Detect hardware

In [ ]:
import torch, jax
HARDWARE = 'tpu' if any('tpu' in str(d).lower() for d in jax.devices()) else ('gpu' if torch.cuda.is_available() else 'cpu')
DEVICE = {'tpu':'tpu','gpu':'cuda','cpu':'cpu'}[HARDWARE]
HOURLY = {'tpu': 1.20, 'gpu': 0.71, 'cpu': 0.05}[HARDWARE]  # rough on-demand
print(HARDWARE, DEVICE)

## 3. Load model

In [ ]:
from vllm import LLM, SamplingParams
import time
t0 = time.time()
MODEL = 'google/gemma-2-2b'   # or local path: './out_tpu' / './out_gpu'
llm = LLM(model=MODEL, device=DEVICE, max_model_len=2048, seed=42, trust_remote_code=True)
load_time = time.time() - t0
print(f'load: {load_time:.1f}s')

## 4. Latency + throughput

In [ ]:
import statistics, time
PROMPTS = ['Write Fibonacci in Python.', 'Explain monads in 3 sentences.',
           'Top SQL query for top-5 customers by sales.', 'Haiku about TPUs.',
           'Translate "hello" to Japanese.', 'Regex for IPv4.',
           'Big-O of merge sort?', 'Define Pareto principle.']
prompts = [PROMPTS[i % len(PROMPTS)] for i in range(128)]
sp = SamplingParams(temperature=0.0, max_tokens=256, seed=42)

# warmup
llm.generate(prompts[:8], sp)

lats = []
for p in prompts[:16]:
    t0 = time.time(); llm.generate([p], sp); lats.append((time.time()-t0)*1000)

t0 = time.time()
outs = llm.generate(prompts, sp)
total_time = time.time() - t0
total_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
throughput = total_tokens / total_time
cost_per_1m = (total_time/3600) * HOURLY / (total_tokens / 1_000_000)

metrics = {
  'model': MODEL, 'hardware': HARDWARE, 'device': DEVICE,
  'load_time_sec': round(load_time,2),
  'total_prompts': len(prompts), 'total_tokens': total_tokens,
  'total_time_sec': round(total_time,3),
  'throughput_tokens_per_sec': round(throughput,2),
  'latency_p50_ms': round(statistics.median(lats),2),
  'latency_p95_ms': round(max(lats),2),
  'hourly_cost_usd': HOURLY,
  'cost_per_1m_tokens_usd': round(cost_per_1m,4),
}
if HARDWARE=='gpu':
    metrics['peak_memory_gb'] = round(torch.cuda.max_memory_allocated()/1e9, 3)
import json
print(json.dumps(metrics, indent=2))

## 5. Save

In [ ]:
from pathlib import Path
Path('benchmarks/results').mkdir(parents=True, exist_ok=True)
fname = f'benchmarks/results/{HARDWARE}_vllm_{int(time.time())}.json'
Path(fname).write_text(json.dumps({'name': f'{HARDWARE}-{MODEL}', 'metrics': metrics}, indent=2))
print(fname)